# D2 Fast-LoRA: бесплатная Kaggle T4, одна GPU

Это **не выполненный эксперимент**. Все веса и токены — **SYNTHETIC**; модели с Hub не
скачиваются. Notebook проверяет существующий harness и сохраняет реальные результаты,
включая ошибки. Он не публикует notebook, не загружает архив и не изменяет Trello.
Правило: `benchmarks/gate-d2-fast-lora-rule.md` из проверенного checkout.

1. Импортируйте этот файл в **private** Kaggle Notebook. Начните с **Accelerator: None**,
   **Internet: On** в интерфейсе Kaggle. Установка и CPU-проверки не требуют активной GPU.
2. Выполните разделы 1–5 с `RUN_CUDA = False`. Допустим только Python **3.10–3.12**.
   Неподдерживаемый образ останавливается **до установки**; не обходите ограничение проекта.
3. Когда CPU-проверки зелёные и вы готовы расходовать бесплатную квоту, вручную выберите
   **GPU T4** и Internet в Kaggle. Даже если интерфейс выделил две T4, процесс видит только
   физическую GPU 0: `CUDA_VISIBLE_DEVICES=0`. Вторая GPU не используется.
4. Установите `RUN_CUDA = True` и выполните разделы по порядку. Если Kaggle перезапустил
   сессию, выполните notebook с начала. Кэш не является доказательством прошедших проверок.
   Перед перезапуском скачайте нужные CPU-артефакты.
5. При первой ошибке остановитесь. Не меняйте seed, dtype, 50 шагов или допуски, чтобы
   скрыть несовпадение. Ошибка градиентов — FAIL, а не повод попросить более дорогую GPU.
   Последнюю ячейку архива можно выполнить отдельно после ошибки.

По умолчанию используется **private frozen source ZIP**, а не неопубликованная GitHub-ветка.
Владелец после freeze/review собирает `d2-source-*.zip` с файлами репозитория в корне и
`D2-SNAPSHOT.json`; предыдущий handoff ZIP с `source/evidence/` не подходит. Добавьте его
как **private Kaggle Input** вручную. Никакие аккаунты, логины или upload notebook не создаёт.
В разделе 1 укажите точный `SOURCE_ZIP` и сравните/вставьте хеш владельца в
`EXPECTED_ZIP_SHA256`. Пустой путь разрешает только **единственный** `d2-source-*.zip`
под `/kaggle/input`; отсутствие/несколько файлов — остановка, не Git fallback.
Сверка хеша рекомендуется: внутренний manifest подтверждает целостность, не автора.
`SOURCE_MODE="git"` — только явный необязательный legacy opt-in к уже публичной ветке;
этот notebook не разрешает push и никогда сам не переключается на `main`.


## 1. Параметры и ограничения

По умолчанию — private ZIP, подготовка и CPU. Мини-тайминг CUDA разрешён только после
всех соответствующих проверок. Большой режим — **один 8B-shaped блок**, не полная
8B-модель; он выключен. Каждая конфигурация получает **новый** `WORK_SOURCE` в
`/kaggle/working`; старые source/run/venv папки не используются, не очищаются и не удаляются.
ZIP input остаётся read-only. Неверный path/hash/manifest блокирует всё до установки
и исполнения исходного кода. Не распаковывайте старый handoff archive вместо source ZIP.


In [ ]:
import csv
import hashlib
import importlib.metadata
import json
import math
import os
import re
import shutil
import stat
import subprocess
import sys
import time
import uuid
import xml.etree.ElementTree as ET
import zipfile
from datetime import datetime, timezone
from pathlib import Path

# Set before ANY torch import; this kernel never imports torch itself.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
if not __debug__:
    raise RuntimeError("Отключённые Python assertions запрещены; используйте обычный kernel.")
if "torch" in sys.modules:
    raise RuntimeError("Перезапустите чистый kernel: torch уже импортирован до выбора GPU.")
if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError(
        f"Python {sys.version.split()[0]} не поддерживается: проект требует 3.10–3.12. "
        "Выберите совместимый Kaggle image; установка не запускалась."
    )
WORK = Path("/kaggle/working")
if not WORK.is_dir() or sys.platform != "linux":
    raise RuntimeError("Этот notebook запускается только в Linux Kaggle, не локально.")

RUN_CUDA = False
RUN_TINY_TIMING = True
RUN_8B_SHAPED_BLOCK = False
REPO_URL = "https://github.com/SoupTeam/soup-rnd.git"
BRANCH = "d2/fast-lora-correctness"
SOURCE_MODE = "zip"  # Only explicit "git" opts into the old public branch.
SOURCE_ZIP = ""  # Exact /kaggle/input/.../d2-source-....zip; blank = unique discovery.
EXPECTED_ZIP_SHA256 = ""  # Compare against the frozen builder hash before running.
INPUT_ROOT = Path("/kaggle/input")
SEED = 792
STEPS = 50
WARMUP = 5
REPEATS = 5
TOKENS = 16
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
WORK_SOURCE = WORK / ("d2-source-" + RUN_ID)
CHECKOUT = WORK_SOURCE
RUN = WORK / "d2-evidence" / RUN_ID
RUN.mkdir(parents=True, exist_ok=False)
VENV = WORK / "d2-kaggle-env" / RUN_ID
PYTHON = sys.executable
GATES = {
    "CPU_GRADCHECK": {"status": "UNRUN"},
    "CPU_PARITY": {"status": "UNRUN"},
    "CPU_REGRESSIONS": {"status": "UNRUN"},
    "CPU_NF4_CHECKPOINT": {"status": "UNRUN"},
    "CPU_MIXED_PRECISION": {"status": "UNRUN"},
    "CPU_REFERENCE_PRECISION": {"status": "UNRUN"},
    "APPLE_MPS": {
        "status": "UNVERIFIED_PLATFORM_BLOCK",
        "reason": "Kaggle Linux excludes Apple MPS cases with -k 'not mps'",
    },
    "CUDA_PREFLIGHT": {"status": "UNVERIFIED", "reason": "RUN_CUDA=False by default"},
    "CUDA_DENSE_PARITY": {"status": "UNVERIFIED"},
    "CUDA_LOSS_50": {"status": "UNVERIFIED"},
    "CUDA_NF4_SINGLE_CHECKPOINT_FP16": {"status": "UNVERIFIED"},
    "CUDA_NATIVE_BF16": {
        "status": "UNVERIFIED_HARDWARE_BLOCK",
        "reason": "T4 compute capability 7.5: no native BF16; fp16 is not a substitute",
    },
    "CUDA_TRITON_LIGER_UNSLOTH": {
        "status": "UNVERIFIED_HARDWARE_BLOCK",
        "reason": "T4 is outside current Triton NVIDIA compute capability 8.0+ support",
    },
    "CUDA_NF4_QKV_MLP_BF16": {
        "status": "UNVERIFIED_HARDWARE_BLOCK",
        "reason": "Existing QKV/MLP GPU NF4 cases request bf16; not run on T4",
    },
    "CUDA_NF4_STREAMED": {"status": "UNVERIFIED", "reason": "No CUDA NF4 streamed evidence"},
    "HARNESS_NF4": {"status": "UNVERIFIED", "reason": "Harness implements dense only"},
    "CUDA_TINY_TIMING": {"status": "UNVERIFIED"},
    "CUDA_8B_SHAPED_BLOCK": {"status": "UNVERIFIED", "reason": "Explicit opt-in disabled"},
}
(RUN / "configuration.json").write_text(
    json.dumps(
        {
            "run_id": RUN_ID,
            "source_mode": SOURCE_MODE,
            "source_zip": SOURCE_ZIP,
            "expected_zip_sha256": EXPECTED_ZIP_SHA256,
            "repository": REPO_URL if SOURCE_MODE == "git" else None,
            "branch": BRANCH if SOURCE_MODE == "git" else None,
            "checkout": str(CHECKOUT),
            "run_cuda": RUN_CUDA,
            "tiny_timing": RUN_TINY_TIMING,
            "run_8b_shaped_block": RUN_8B_SHAPED_BLOCK,
            "seed": SEED,
            "loss_steps": STEPS,
            "warmup": WARMUP,
            "repeats": REPEATS,
            "tokens": TOKENS,
            "kernel_python": sys.version,
            "cuda_visible_devices": "0",
            "publication": "disabled",
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

## 2. Журнал команд

Каждый запуск получает отдельную папку. Полные stdout/stderr пишутся прямо в файлы,
код возврата проверяется, а `commands.jsonl` сохраняет аргументы и разрешённые поля
окружения (не все переменные окружения). Timeout/прерывание тоже сохраняются.
Ни одна пропущенная GPU-проверка не получает PASS. Сбой `nvidia-smi` не скрывается.

In [ ]:
def say(message: str) -> None:
    try:
        from rich.console import Console

        Console().print(message, markup=False, highlight=False)
    except ImportError:
        sys.stderr.write(message + "\n")


def write_json(path: Path, value: object) -> None:
    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")


def save_gates() -> None:
    write_json(RUN / "gate_status.json", GATES)


class UnverifiedGateError(RuntimeError):
    pass


def artifact_dir(label: str) -> Path:
    if not re.fullmatch(r"[a-z0-9_-]+", label):
        raise ValueError(f"Unsafe artifact label: {label}")
    path = RUN / (label + "-" + uuid.uuid4().hex[:8])
    path.mkdir(exist_ok=False)
    return path


def child_env(device: str) -> dict[str, str]:
    if device not in {"cpu", "cuda"}:
        raise ValueError(device)
    env = dict(os.environ)
    env.update(
        {
            "CUDA_VISIBLE_DEVICES": "0" if device == "cuda" else "",
            "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
            "PYTHONUNBUFFERED": "1",
            "PYTHONDONTWRITEBYTECODE": "1",
            "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
            "OMP_NUM_THREADS": "1",
            "MKL_NUM_THREADS": "1",
            "TOKENIZERS_PARALLELISM": "false",
            "HF_HUB_OFFLINE": "1",
            "TRANSFORMERS_OFFLINE": "1",
            "HF_DATASETS_OFFLINE": "1",
            "HF_HUB_DISABLE_TELEMETRY": "1",
            "WANDB_MODE": "disabled",
            "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
            "PYTEST_ADDOPTS": "",
            "PYTHONOPTIMIZE": "0",
            "PYTHONHASHSEED": str(SEED),
        }
    )
    if CHECKOUT.is_dir():
        env["PYTHONPATH"] = str(CHECKOUT / "src")
    else:
        env.pop("PYTHONPATH", None)
    return env


def run_logged(
    label: str,
    argv: list[str],
    *,
    device: str = "cpu",
    cwd: Path | None = None,
    directory: Path | None = None,
    timeout: int = 1800,
    strict: bool = True,
) -> dict:
    if globals().get("SOURCE_MODE") == "zip" and "CHECKOUT_INFO" in globals():
        assert_checkout()  # Hash/inventory guard BEFORE any source/install subprocess.
    directory = directory or artifact_dir(label)
    env = child_env(device)
    record = {
        "label": label,
        "argv": [str(part) for part in argv],
        "cwd": str(cwd or WORK),
        "directory": str(directory),
        "utc_start": datetime.now(timezone.utc).isoformat(),
        "timeout_seconds": timeout,
        "environment": {
            key: env.get(key)
            for key in (
                "CUDA_VISIBLE_DEVICES",
                "CUDA_DEVICE_ORDER",
                "PYTHONPATH",
                "CUBLAS_WORKSPACE_CONFIG",
                "HF_HUB_OFFLINE",
                "PYTEST_DISABLE_PLUGIN_AUTOLOAD",
                "PYTHONOPTIMIZE",
                "PYTHONHASHSEED",
            )
        },
        "returncode": None,
        "status": "RUNNING",
    }
    write_json(directory / "command.json", record)
    started = time.monotonic()
    caught = None
    try:
        with (directory / "stdout.log").open("wb") as stdout_file:
            with (directory / "stderr.log").open("wb") as stderr_file:
                process = subprocess.run(
                    record["argv"],
                    cwd=str(cwd or WORK),
                    env=env,
                    stdout=stdout_file,
                    stderr=stderr_file,
                    check=False,
                    timeout=timeout,
                )
        record["returncode"] = process.returncode
        record["status"] = "EXIT_ZERO" if process.returncode == 0 else "FAILED_COMMAND"
        if globals().get("SOURCE_MODE") == "zip" and "CHECKOUT_INFO" in globals():
            try:
                assert_checkout()
            except BaseException as exc:
                caught = exc
                record["status"] = "FAILED_SOURCE_SNAPSHOT"
                record["error"] = f"{type(exc).__name__}: {exc}"
    except BaseException as exc:
        caught = exc
        record["status"] = "INTERRUPTED_OR_LAUNCH_FAILED"
        record["error"] = f"{type(exc).__name__}: {exc}"
    finally:
        record["elapsed_seconds"] = time.monotonic() - started
        record["utc_end"] = datetime.now(timezone.utc).isoformat()
        write_json(directory / "command.json", record)
        with (RUN / "commands.jsonl").open("a", encoding="utf-8") as ledger:
            ledger.write(json.dumps(record, allow_nan=False) + "\n")
    say(f"{label}: {record['status']} (exit={record['returncode']}); {directory}")
    if caught is not None and (strict or isinstance(caught, KeyboardInterrupt)):
        raise caught
    if (record["returncode"] != 0 or record["status"] != "EXIT_ZERO") and strict:
        for filename in ("stdout.log", "stderr.log"):
            path = directory / filename
            if path.exists():
                say(path.read_text(encoding="utf-8", errors="replace")[-6000:])
        raise RuntimeError(f"Команда {label} завершилась с ошибкой; полные логи: {directory}")
    return record


def run_text(label: str, argv: list[str], *, cwd: Path | None = None) -> str:
    result = run_logged(label, argv, cwd=cwd)
    return (Path(result["directory"]) / "stdout.log").read_text(encoding="utf-8").strip()


CORE_TESTS = [
    "tests/test_issue839_fast_lora_single_projection.py",
    "tests/test_issue838_fast_lora_qkv.py",
    "tests/test_issue837_fast_lora_mlp.py",
]
D2_TESTS = [
    "tests/test_d2_qkv_backward.py",
    "tests/test_d2_checkpoint_and_dtypes.py",
    "tests/test_d2_qkv_semantics.py",
    "tests/test_d2_fast_lora_probe.py",
    "tests/test_d2_mixed_precision_backward.py",
    "tests/test_d2_reference_precision.py",
]
VALIDITY_TESTS = [
    "tests/test_d2_fast_lora_probe_validity.py",
    "tests/test_d2_kaggle_notebook_validity.py",
]
TEST_FILES = CORE_TESTS + D2_TESTS + VALIDITY_TESTS + ["tests/test_d2_kaggle_notebook.py"]
KERNEL_FILES = ["fast_lora.py", "fast_lora_qkv.py", "fast_lora_mlp.py"]
REQUIRED_SOURCE_FILES = [
    "pyproject.toml",
    "README.md",
    "LICENSE",
    "AGENTS.md",
    "CONTRIBUTING.md",
    "src/soup_cli/__init__.py",
    "tests/__init__.py",
    "tests/conftest.py",
    "notebooks/d2-fast-lora-kaggle.ipynb",
    "benchmarks/d2-kaggle-runbook.md",
    "benchmarks/gate-d2-fast-lora-rule.md",
    "benchmarks/harness/fast_lora_probe.py",
    *("src/soup_cli/utils/" + name for name in KERNEL_FILES),
    *TEST_FILES,
]
SOURCE_FILES = REQUIRED_SOURCE_FILES


def safe_artifact_path(path: Path, root: Path) -> os.stat_result:
    # Never resolve+relative_to: realpath/commonpath handles Windows short names.
    if ".." in path.parts or ".." in root.parts:
        raise RuntimeError(f"Unsafe artifact traversal: {path}")
    root_real = os.path.normcase(os.path.realpath(root))
    path_real = os.path.normcase(os.path.realpath(path))
    try:
        contained = os.path.commonpath([root_real, path_real]) == root_real
    except ValueError:
        contained = False
    if not contained:
        raise RuntimeError(f"Unsafe artifact outside containment root: {path}")
    # Check ancestors too: an ordinary file can sit behind a symlink/junction.
    absolute = path.absolute()
    for component in (absolute, *absolute.parents):
        metadata = os.lstat(component)
        if getattr(metadata, "st_file_attributes", 0) & getattr(
            stat, "FILE_ATTRIBUTE_REPARSE_POINT", 0x400
        ) or stat.S_ISLNK(metadata.st_mode):
            raise RuntimeError(f"Unsafe artifact link/reparse point: {component}")
    metadata = os.lstat(path)
    if not (stat.S_ISREG(metadata.st_mode) or stat.S_ISDIR(metadata.st_mode)):
        raise RuntimeError(f"Unsafe nonregular artifact: {path}")
    if stat.S_ISREG(metadata.st_mode) and metadata.st_nlink != 1:
        raise RuntimeError(f"Unsafe hardlinked artifact: {path}")
    return metadata


def source_member_name(name: str) -> str:
    if (
        not isinstance(name, str)
        or not name
        or name.startswith("/")
        or "\\" in name
        or ":" in name
        or "\0" in name
        or any(char in '<>"?*|' for char in name)
    ):
        raise RuntimeError("Unsafe source ZIP member name")
    parts = name.split("/")
    reserved = {
        "con",
        "prn",
        "aux",
        "nul",
        "conin$",
        "conout$",
        *(f"{prefix}{suffix}" for prefix in ("com", "lpt") for suffix in "123456789¹²³"),
    }
    credentials = {
        "kaggle.json",
        "credentials.json",
        "credentials",
        "service-account.json",
        "id_rsa",
        "id_ed25519",
        "hf_token",
        "token",
        "access_token",
        "token.json",
        ".netrc",
        ".git-credentials",
        ".npmrc",
        ".pypirc",
    }
    for part in parts:
        lower = part.casefold()
        if (
            part in {"", ".", ".."}
            or part.endswith((".", " "))
            or any(ord(char) < 32 for char in part)
            or lower.split(".", 1)[0].rstrip(" ") in reserved
            or lower in {".git", ".ssh", ".aws", ".kaggle", ".config", "__pycache__"}
            or lower == ".env"
            or lower.startswith(".env.")
            or lower in credentials
            or lower.endswith((".pem", ".key", ".p12", ".pfx", ".pyc"))
        ):
            raise RuntimeError("Unsafe source ZIP member/credential filename")
    return name


def allowed_source_file(name: str) -> bool:
    source_member_name(name)
    return (
        name in REQUIRED_SOURCE_FILES
        or name == "docs/peft-and-efficiency.md"
        or name.startswith("src/soup_cli/")
    )


def source_json(data: bytes) -> dict:
    def unique_pairs(pairs):
        value = {}
        for key, item in pairs:
            if key in value:
                raise RuntimeError("Duplicate snapshot manifest key")
            value[key] = item
        return value

    try:
        value = json.loads(data.decode("utf-8"), object_pairs_hook=unique_pairs)
    except (UnicodeError, ValueError) as exc:
        raise RuntimeError("Unreadable source snapshot manifest") from exc
    if not isinstance(value, dict):
        raise RuntimeError("Source snapshot manifest must be an object")
    return value


def validate_snapshot(snapshot: dict) -> None:
    fields = {
        "format",
        "base_revision",
        "source_sha256",
        "kernel_sha256",
        "harness_sha256",
        "decision_rule_sha256",
        "snapshot_id",
    }
    if set(snapshot) != fields or snapshot.get("format") != "soup-d2-snapshot-v1":
        raise RuntimeError("Unexpected source snapshot manifest schema/format")
    if snapshot["base_revision"] != "9aa43bd71afe12d3724f196202f1140e7dc409dc":
        raise RuntimeError("Unexpected snapshot base_revision metadata (not a Git HEAD claim)")
    hashes = snapshot["source_sha256"]
    if not isinstance(hashes, dict) or not set(REQUIRED_SOURCE_FILES).issubset(hashes):
        raise RuntimeError("Source snapshot missing required inputs")
    aliases = set()
    for name, digest in hashes.items():
        if not allowed_source_file(name) or name.casefold() in aliases:
            raise RuntimeError("Unsafe source snapshot inventory/duplicate member")
        aliases.add(name.casefold())
        if not isinstance(digest, str) or not re.fullmatch(r"[0-9a-f]{64}", digest):
            raise RuntimeError("Invalid snapshot source SHA256")
    expected_kernels = {name: hashes["src/soup_cli/utils/" + name] for name in KERNEL_FILES}
    if snapshot["kernel_sha256"] != expected_kernels:
        raise RuntimeError("Snapshot kernel SHA256 set mismatch")
    if snapshot["harness_sha256"] != hashes["benchmarks/harness/fast_lora_probe.py"]:
        raise RuntimeError("Snapshot harness SHA256 mismatch")
    if snapshot["decision_rule_sha256"] != hashes["benchmarks/gate-d2-fast-lora-rule.md"]:
        raise RuntimeError("Snapshot decision-rule SHA256 mismatch")
    canonical = {key: value for key, value in snapshot.items() if key != "snapshot_id"}
    expected_id = hashlib.sha256(
        json.dumps(
            canonical,
            sort_keys=True,
            separators=(",", ":"),
            ensure_ascii=True,
        ).encode("utf-8")
    ).hexdigest()
    if snapshot["snapshot_id"] != expected_id:
        raise RuntimeError("Source snapshot_id SHA256 mismatch")


def safe_source_bytes(path: Path, root: Path, limit: int = 256 * 1024 * 1024) -> bytes:
    before = safe_artifact_path(path, root)
    if not stat.S_ISREG(before.st_mode) or before.st_size > limit:
        raise RuntimeError("Unsafe source file type/size")
    descriptor = os.open(
        path, os.O_RDONLY | getattr(os, "O_BINARY", 0) | getattr(os, "O_NOFOLLOW", 0)
    )
    with os.fdopen(descriptor, "rb") as handle:
        opened = os.fstat(handle.fileno())
        if (
            not stat.S_ISREG(opened.st_mode)
            or opened.st_nlink != 1
            or (opened.st_dev, opened.st_ino) != (before.st_dev, before.st_ino)
        ):
            raise RuntimeError("Unsafe source file changed before read")
        data = handle.read(limit + 1)
    after = safe_artifact_path(path, root)
    if len(data) > limit or (after.st_dev, after.st_ino, after.st_size, after.st_mtime_ns) != (
        opened.st_dev,
        opened.st_ino,
        opened.st_size,
        opened.st_mtime_ns,
    ):
        raise RuntimeError("Unsafe source file changed during read/size limit")
    return data


def source_parent_names(names) -> set[str]:
    return {
        "/".join(name.split("/")[:index])
        for name in names
        for index in range(1, len(name.split("/")))
    }


def verify_source_tree(root: Path, info: dict) -> None:
    safe_artifact_path(root, WORK)
    snapshot = info["snapshot"]
    validate_snapshot(snapshot)
    expected = set(snapshot["source_sha256"]) | {"D2-SNAPSHOT.json"}
    parents = source_parent_names(expected)
    actual = set()
    directories = set()

    def walk_error(error):
        raise error

    # Check the complete tree BEFORE any source hash/read.
    for directory, subdirectories, filenames in os.walk(
        root, followlinks=False, onerror=walk_error
    ):
        for name in subdirectories + filenames:
            path = Path(directory) / name
            metadata = safe_artifact_path(path, root)
            relative = Path(os.path.relpath(path, root)).as_posix()
            source_member_name(relative)
            (directories if stat.S_ISDIR(metadata.st_mode) else actual).add(relative)
    if actual != expected or directories != parents:
        raise RuntimeError("Frozen source snapshot inventory changed")
    manifest_bytes = safe_source_bytes(root / "D2-SNAPSHOT.json", root, 1024 * 1024)
    if (
        hashlib.sha256(manifest_bytes).hexdigest() != info["manifest_sha256"]
        or source_json(manifest_bytes) != snapshot
    ):
        raise RuntimeError("Frozen source snapshot manifest changed")
    for name, digest in snapshot["source_sha256"].items():
        if hashlib.sha256(safe_source_bytes(root / name, root)).hexdigest() != digest:
            raise RuntimeError("Frozen source snapshot SHA256 changed")


def select_source_zip(configured: str, input_root: Path) -> Path:
    safe_artifact_path(input_root, input_root)
    if configured:
        path = Path(configured)
        if not path.is_absolute() or path.suffix != ".zip" or not path.is_file():
            raise RuntimeError("Configured private source ZIP missing/invalid; no Git fallback")
    else:
        matches = sorted(input_root.rglob("d2-source-*.zip"))
        if len(matches) != 1:
            raise RuntimeError("Private source ZIP discovery requires exactly one d2-source-*.zip")
        path = matches[0]
    safe_artifact_path(path, input_root)
    return path


def assert_checkout() -> dict:
    if SOURCE_MODE == "git":
        return assert_git_checkout()
    verify_source_tree(CHECKOUT, CHECKOUT_INFO)
    snapshot = CHECKOUT_INFO["snapshot"]
    return {
        "source_mode": "zip",
        "git_state": "NO_GIT_SNAPSHOT",
        "head": None,
        "snapshot_id": snapshot["snapshot_id"],
        "base_revision": snapshot["base_revision"],
        "zip_sha256": CHECKOUT_INFO["zip_sha256"],
    }


def extract_source_zip(path: Path, target: Path, expected_sha256: str = "") -> dict:
    import io
    import zipfile

    if expected_sha256 and not re.fullmatch(r"[0-9a-f]{64}", expected_sha256):
        raise RuntimeError("EXPECTED_ZIP_SHA256 must be exactly 64 lowercase hex characters")
    archive_bytes = safe_source_bytes(path, path.parent)
    digest = hashlib.sha256(archive_bytes).hexdigest()
    if expected_sha256 and digest != expected_sha256:
        raise RuntimeError("Private source ZIP SHA256 mismatch")
    # Parse/hash all members in memory first; no source/install executes here.
    with zipfile.ZipFile(io.BytesIO(archive_bytes)) as bundle:
        entries = bundle.infolist()
        if len(entries) > 10000 or sum(entry.file_size for entry in entries) > 256 * 1024 * 1024:
            raise RuntimeError("Source ZIP size/member limit")
        files, directories, aliases = {}, set(), set()
        for entry in entries:
            name = entry.orig_filename
            directory = entry.is_dir()
            name = source_member_name(name[:-1] if directory else name)
            if name.casefold() in aliases:
                raise RuntimeError("Duplicate source ZIP member/alias")
            aliases.add(name.casefold())
            mode = stat.S_IFMT(entry.external_attr >> 16)
            if (
                entry.flag_bits & 1
                or entry.external_attr & 0x400
                or mode not in {0, stat.S_IFDIR if directory else stat.S_IFREG}
                or entry.file_size > 64 * 1024 * 1024
            ):
                raise RuntimeError("Unsafe nonregular/encrypted source ZIP member")
            if directory:
                directories.add(name)
            else:
                if name != "D2-SNAPSHOT.json" and not allowed_source_file(name):
                    raise RuntimeError("Unsafe source ZIP inventory member")
                files[name] = entry
        if "D2-SNAPSHOT.json" not in files or files["D2-SNAPSHOT.json"].file_size > 1024 * 1024:
            raise RuntimeError("Source snapshot manifest missing/oversized")
        manifest_bytes = bundle.read(files["D2-SNAPSHOT.json"])
        snapshot = source_json(manifest_bytes)
        validate_snapshot(snapshot)
        expected = set(snapshot["source_sha256"]) | {"D2-SNAPSHOT.json"}
        parents = source_parent_names(expected)
        if set(files) != expected or not directories.issubset(parents) or set(files) & parents:
            raise RuntimeError("Source ZIP inventory/parent conflict")
        data = {name: bundle.read(entry) for name, entry in files.items()}
        for name, expected_hash in snapshot["source_sha256"].items():
            if hashlib.sha256(data[name]).hexdigest() != expected_hash:
                raise RuntimeError("Private source file SHA256 mismatch")
    safe_artifact_path(target.parent, WORK)
    if ".." in target.parts or os.path.lexists(target):
        raise RuntimeError("Unsafe/existing WORK_SOURCE; never reuse or delete old work")
    target_real = os.path.normcase(os.path.realpath(target))
    work_real = os.path.normcase(os.path.realpath(WORK))
    if os.path.commonpath([target_real, work_real]) != work_real:
        raise RuntimeError("Unsafe WORK_SOURCE outside containment root")
    target.mkdir(exist_ok=False)
    for name, value in data.items():
        destination = target / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        safe_artifact_path(destination.parent, target)
        with destination.open("xb") as handle:
            handle.write(value)
    info = {
        "snapshot": snapshot,
        "zip_sha256": digest,
        "zip_path": str(path),
        "manifest_sha256": hashlib.sha256(manifest_bytes).hexdigest(),
        "operator_hash_verified": bool(expected_sha256),
        "git_state": "NO_GIT_SNAPSHOT",
    }
    verify_source_tree(target, info)
    return info


GATE_DEPENDENCIES = {
    "CPU_GRADCHECK": (),
    "CPU_PARITY": ("CPU_GRADCHECK",),
    "CPU_REGRESSIONS": ("CPU_PARITY",),
    "CPU_NF4_CHECKPOINT": ("CPU_REGRESSIONS",),
    "CPU_MIXED_PRECISION": ("CPU_NF4_CHECKPOINT",),
    "CPU_REFERENCE_PRECISION": ("CPU_MIXED_PRECISION",),
    "CUDA_PREFLIGHT": ("CPU_REFERENCE_PRECISION",),
    "CUDA_DENSE_PARITY": ("CUDA_PREFLIGHT",),
    "CUDA_LOSS_50": ("CUDA_DENSE_PARITY",),
    "CUDA_NF4_SINGLE_CHECKPOINT_FP16": ("CUDA_LOSS_50",),
    "CUDA_TINY_TIMING": ("CUDA_NF4_SINGLE_CHECKPOINT_FP16",),
    "CUDA_8B_SHAPED_BLOCK": ("CUDA_NF4_SINGLE_CHECKPOINT_FP16",),
}


def set_gate_status(name: str, result: dict) -> None:
    previous = GATES.get(name, {})
    history = list(previous.get("history", []))
    if previous:
        history.append({key: value for key, value in previous.items() if key != "history"})
    GATES[name] = {**result, "history": history, "utc": datetime.now(timezone.utc).isoformat()}


def begin_gate(name: str) -> None:
    invalidated = {name}
    while True:
        descendants = {
            child
            for child, parents in GATE_DEPENDENCIES.items()
            if invalidated.intersection(parents)
        } - invalidated
        if not descendants:
            break
        invalidated.update(descendants)
    for child in sorted(invalidated - {name}):
        set_gate_status(
            child,
            {
                "status": "UNVERIFIED",
                "invalidated_by": name,
                "reason": "Upstream retry requires fresh evidence",
            },
        )
    set_gate_status(name, {"status": "RUNNING"})
    save_gates()


def require_gates(*names: str) -> None:
    checked = set()
    visiting = set()

    def check(name: str) -> None:
        if name in visiting:
            raise UnverifiedGateError(f"Cyclic gate dependency: {name}")
        if name in checked:
            return
        result = GATES.get(name, {})
        if result.get("status") != "PASS":
            raise UnverifiedGateError(f"{name} не PASS: {result}")
        visiting.add(name)
        for parent in GATE_DEPENDENCIES.get(name, ()):
            check(parent)
        visiting.remove(name)
        checked.add(name)

    for name in names:
        check(name)


def gate(name: str, action) -> object:
    # Invalidate descendants BEFORE every attempt, even if the retry later passes.
    begin_gate(name)
    try:
        require_gates(*GATE_DEPENDENCIES.get(name, ()))
        value = action()
    except BaseException as exc:
        set_gate_status(
            name,
            {
                "status": "UNVERIFIED" if isinstance(exc, UnverifiedGateError) else "FAIL",
                "error": f"{type(exc).__name__}: {exc}",
            },
        )
        save_gates()
        raise
    set_gate_status(name, {"status": "PASS"})
    save_gates()
    return value


def smi_snapshot(label: str) -> dict:
    queries = {
        "cards": ["nvidia-smi", "-L"],
        "state": [
            "nvidia-smi",
            "--query-gpu=index,uuid,name,driver_version,pstate,"
            "clocks.current.sm,clocks.current.memory,power.draw,power.limit,"
            "temperature.gpu,utilization.gpu,memory.used,memory.total",
            "--format=csv,noheader,nounits",
        ],
        "peers": [
            "nvidia-smi",
            "--query-compute-apps=gpu_uuid,pid,process_name,used_memory",
            "--format=csv,noheader,nounits",
        ],
    }
    records = {
        name: run_logged(label + "-" + name, argv, device="cuda", strict=False, timeout=15)
        for name, argv in queries.items()
    }
    write_json(artifact_dir(label + "-snapshot") / "snapshot.json", records)
    return records


save_gates()
say(f"Артефакты текущего запуска: {RUN}")

## 3. Проверенный private source ZIP и неизменённый PyTorch

Сначала проверяются весь ZIP inventory, безопасные root-relative имена, точная схема
`D2-SNAPSHOT.json`, полный SHA256 map и обязательные входы. Запрещены ссылки, special files,
case aliases, `..`, absolute/UNC/drive/backslash/NUL, `.git`, `.env` и реальные credential
filenames. `tokenizers.py` — допустимый source filename. Никакой код из ZIP не импортируется
до проверки всех байтов. После этого файлы записываются только в новый `WORK_SOURCE`;
проверки повторяются перед каждым subprocess. Подробный контракт сборки — в runbook.

`base_revision` — историческая база, **не** выдуманный Git HEAD. ZIP работает без `.git`;
snapshot ID, ZIP/source/kernel/harness/rule hashes — обязательное происхождение.


Создаётся отдельный venv с `--system-site-packages`: используются wheel/build Torch из
Kaggle и их исходный путь. Они фиксируются constraint и проверяются повторно. Нет
`[dev]`/`[all]`, полного TRL-стека или установки нового Torch. Версии прямых зависимостей
зафиксированы ниже, полный фактический набор сохраняется через `pip freeze --all`.
Транзитивные зависимости разрешает pip; это не lockfile всей Kaggle-системы.

Если current Torch отсутствует, старее 2.6, версии 3.x или не импортируется, notebook
останавливается. Не переустанавливайте его вслепую. Смена Kaggle image — ручная операция,
а не автоматическая «починка» результата. Несовместимый torchvision/аудиопакет также
должен дать настоящий import error, а не быть незаметно удалён.

In [ ]:
def assert_git_checkout() -> dict:
    actual_root = run_text("git-root", ["git", "rev-parse", "--show-toplevel"], cwd=CHECKOUT)
    if os.path.realpath(actual_root) != os.path.realpath(CHECKOUT):
        raise RuntimeError("CHECKOUT не является корнем отдельного репозитория.")
    origin = run_text("git-origin", ["git", "remote", "get-url", "origin"], cwd=CHECKOUT)
    branch = run_text("git-branch", ["git", "branch", "--show-current"], cwd=CHECKOUT)
    head = run_text("git-head", ["git", "rev-parse", "HEAD"], cwd=CHECKOUT)
    dirty = run_text(
        "git-status", ["git", "status", "--porcelain", "--untracked-files=all"], cwd=CHECKOUT
    )
    if origin != REPO_URL or branch != BRANCH or head != REMOTE_HEAD or dirty:
        raise RuntimeError(
            f"Неверный/грязный checkout: origin={origin}, branch={branch}, HEAD={head}, "
            f"expected={REMOTE_HEAD}, status={dirty!r}. Выберите свежий CHECKOUT; "
            "существующие файлы не изменялись и не очищались."
        )
    return {"origin": origin, "branch": branch, "head": head, "status": dirty}


try:
    if SOURCE_MODE == "zip":
        selected_zip = select_source_zip(SOURCE_ZIP, INPUT_ROOT)
        CHECKOUT_INFO = extract_source_zip(selected_zip, WORK_SOURCE, EXPECTED_ZIP_SHA256)
        write_json(RUN / "input-snapshot.json", CHECKOUT_INFO)
        if not EXPECTED_ZIP_SHA256:
            say("ZIP SHA256 сохранён; без сравнения с хешем владельца подлинность не подтверждена.")
    elif SOURCE_MODE == "git":
        try:
            remote = run_text(
                "remote-branch",
                ["git", "ls-remote", "--exit-code", REPO_URL, "refs/heads/" + BRANCH],
            )
        except RuntimeError as exc:
            raise RuntimeError(
                "Публичная D2-ветка недоступна: проверьте Internet в Kaggle UI и push владельца. "
                "Нет автоматического перехода на другую ветку; исходная ошибка в logs."
            ) from exc
        parts = remote.split()
        if len(parts) != 2 or parts[1] != "refs/heads/" + BRANCH:
            raise RuntimeError(
                "Удалённая D2-ветка не найдена; владелец должен сначала отправить её."
            )
        REMOTE_HEAD = parts[0]
        if not re.fullmatch(r"[0-9a-f]{40}", REMOTE_HEAD):
            raise RuntimeError("Некорректный SHA удалённой ветки.")
        if CHECKOUT.is_symlink():
            raise RuntimeError("CHECKOUT является symlink; выберите новый обычный путь.")
        if not CHECKOUT.exists():
            run_logged(
                "clone",
                [
                    "git",
                    "clone",
                    "--depth",
                    "1",
                    "--single-branch",
                    "--branch",
                    BRANCH,
                    REPO_URL,
                    str(CHECKOUT),
                ],
            )
        CHECKOUT_INFO = assert_git_checkout()
    else:
        raise RuntimeError("SOURCE_MODE must be exactly zip or git; no automatic fallback")
except BaseException as exc:
    write_json(
        RUN / "source-preparation.json",
        {
            "status": "FAIL",
            "source_mode": SOURCE_MODE,
            "error": f"{type(exc).__name__}: {exc}",
            "utc": datetime.now(timezone.utc).isoformat(),
        },
    )
    raise
else:
    write_json(
        RUN / "source-preparation.json",
        {
            "status": "PASS",
            "source_mode": SOURCE_MODE,
            "input": CHECKOUT_INFO,
            "utc": datetime.now(timezone.utc).isoformat(),
        },
    )


try:
    ORIGINAL_TORCH_VERSION = importlib.metadata.version("torch")
except importlib.metadata.PackageNotFoundError as exc:
    raise RuntimeError("Kaggle Torch отсутствует; выберите совместимый образ вручную.") from exc
match = re.match(r"^(\d+)\.(\d+)\.(\d+)", ORIGINAL_TORCH_VERSION)
if match is None or not (2, 6, 0) <= tuple(map(int, match.groups())) < (3, 0, 0):
    raise RuntimeError(f"Неподдерживаемый Kaggle Torch: {ORIGINAL_TORCH_VERSION}; нужен >=2.6,<3.")
CPU_TORCH_PROBE = r"""
import json
import os
import sys
from pathlib import Path
from rich.console import Console

def main():
    import torch
    value = {
        "version": torch.__version__, "file": os.path.realpath(torch.__file__),
        "cuda_build": torch.version.cuda, "python": sys.version,
        "device_count": torch.cuda.device_count(),
    }
    assert os.environ["CUDA_VISIBLE_DEVICES"] == ""
    assert value["device_count"] == 0, "CPU preparation must hide every GPU"
    assert torch.ones(2).sum().item() == 2
    Path(sys.argv[1]).write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
    Console().print(value)

main()
"""
original_probe = RUN / "original-kaggle-torch.json"
run_logged("original-torch-cpu", [sys.executable, "-c", CPU_TORCH_PROBE, str(original_probe)])
ORIGINAL_TORCH = json.loads(original_probe.read_text(encoding="utf-8"))
CONSTRAINTS = RUN / "preserve-kaggle-torch.txt"
CONSTRAINTS.write_text("torch==" + ORIGINAL_TORCH_VERSION + "\n", encoding="utf-8")
VENV.parent.mkdir(parents=True, exist_ok=True)
if VENV.exists():
    raise RuntimeError("Новый venv-путь уже существует; запустите конфигурацию для нового RUN_ID.")
run_logged("create-venv", [sys.executable, "-m", "venv", "--system-site-packages", str(VENV)])
PYTHON = str(VENV / "bin" / "python")
PINS = [
    "hatchling==1.27.0",
    "pytest==8.3.5",
    "ruff==0.11.13",
    "pytest-cov==6.1.1",
    "transformers==5.17.0",
    "peft==0.20.0",
    "accelerate==1.14.0",
    "bitsandbytes==0.50.1",
]
write_json(RUN / "direct-dependency-pins.json", PINS)
run_logged(
    "build-backend",
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        "--constraint",
        str(CONSTRAINTS),
        PINS[0],
    ],
)
run_logged(
    "editable-and-test-deps",
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        "--constraint",
        str(CONSTRAINTS),
        "--no-build-isolation",
        "--editable",
        str(CHECKOUT),
        *PINS[1:],
    ],
)
run_logged("pip-freeze", [PYTHON, "-m", "pip", "freeze", "--all"])
# The inherited Kaggle image can have unrelated conflicts. Record, do not call them clean.
GLOBAL_PIP_CHECK = run_logged("pip-check-global", [PYTHON, "-m", "pip", "check"], strict=False)
write_json(RUN / "global-pip-check-status.json", GLOBAL_PIP_CHECK)
installed_probe = RUN / "installed-torch.json"
run_logged("installed-torch-cpu", [PYTHON, "-c", CPU_TORCH_PROBE, str(installed_probe)])
INSTALLED_TORCH = json.loads(installed_probe.read_text(encoding="utf-8"))
if any(INSTALLED_TORCH[key] != ORIGINAL_TORCH[key] for key in ("version", "file", "cuda_build")):
    raise RuntimeError("Torch version/build/import path changed; evidence run must stop.")

## 4. Проверка import path и происхождения

`PYTHONPATH` явно указывает на `CHECKOUT/src` (в ZIP-режиме `CHECKOUT = WORK_SOURCE`).
Import smoke выполняется в CPU-процессе; kernel не занимает GPU. `source-manifest.json`
сохраняет полный source inventory/SHA256, оригинальный input manifest/hash, snapshot ID,
base revision, честное `NO_GIT_SNAPSHOT` и версии. Правило копируется в evidence.
В Git-режиме отдельно проверяются действительные origin/ветка/HEAD/чистота. Нет подмены
отсутствующего Git HEAD метаданными base revision. Любой недостающий обязательный тест
блокирует запуск, а не уменьшает набор. Reference-precision guard проверяет **установленные**
источники Transformers/PEFT и не требует Git-истории ZIP.


In [ ]:
HARNESS = CHECKOUT / "benchmarks/harness/fast_lora_probe.py"
RULE = CHECKOUT / "benchmarks/gate-d2-fast-lora-rule.md"
for relative in SOURCE_FILES:
    if not (CHECKOUT / relative).is_file():
        raise RuntimeError(f"Обязательный файл отсутствует: {relative}")
MANIFEST = {
    **assert_checkout(),
    "utc": datetime.now(timezone.utc).isoformat(),
    "input": CHECKOUT_INFO,
    "files_sha256": {
        relative: hashlib.sha256((CHECKOUT / relative).read_bytes()).hexdigest()
        for relative in (
            CHECKOUT_INFO["snapshot"]["source_sha256"] if SOURCE_MODE == "zip" else SOURCE_FILES
        )
    },
    "test_files": TEST_FILES,
    "original_torch": ORIGINAL_TORCH,
}
write_json(RUN / "source-manifest.json", MANIFEST)
shutil.copyfile(RULE, RUN / "preregistered-rule.md")
IMPORT_PROBE = r"""
import importlib.metadata
import json
import os
import sys
from pathlib import Path
from rich.console import Console

def main():
    import torch
    import peft
    import bitsandbytes
    import soup_cli
    from transformers import LlamaConfig, LlamaForCausalLM
    from soup_cli.utils import fast_lora, fast_lora_qkv, fast_lora_mlp

    root = Path(sys.argv[1])
    modules = (soup_cli, fast_lora, fast_lora_qkv, fast_lora_mlp)
    source_root = os.path.realpath(root / "src")
    for module in modules:
        imported = os.path.realpath(module.__file__)
        assert os.path.commonpath([source_root, imported]) == source_root, imported
    assert os.path.realpath(soup_cli.__file__) == os.path.realpath(
        root / "src/soup_cli/__init__.py"
    ), soup_cli.__file__
    assert torch.cuda.device_count() == 0
    model = LlamaForCausalLM(LlamaConfig(
        vocab_size=32, hidden_size=8, intermediate_size=16, num_hidden_layers=1,
        num_attention_heads=2, num_key_value_heads=1,
    ))
    assert model(torch.zeros((1, 3), dtype=torch.long)).logits.shape == (1, 3, 32)
    versions = {name: importlib.metadata.version(name) for name in (
        "soup-cli", "torch", "transformers", "peft", "bitsandbytes", "accelerate",
        "pytest", "pytest-cov", "ruff", "hatchling",
    )}
    report = {
        "python": sys.version, "executable": sys.executable, "versions": versions,
        "torch_cuda_build": torch.version.cuda,
        "imports": {module.__name__: os.path.realpath(module.__file__) for module in modules},
        "fixture": "SYNTHETIC; no pretrained checkpoint or remote model",
    }
    Path(sys.argv[2]).write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    Console().print(report)

main()
"""
run_logged("import-smoke", [PYTHON, "-c", IMPORT_PROBE, str(CHECKOUT), str(RUN / "versions.json")])
run_logged(
    "targeted-ruff", [PYTHON, "-m", "ruff", "check", "--no-cache", str(HARNESS), *TEST_FILES], cwd=CHECKOUT
)
say(RULE.read_text(encoding="utf-8"))

## 5. CPU float64 gradcheck → dense parity → регрессии и NF4

Сначала проверяются все три существующих gradcheck-набора и harness parity (CPU fp32,
с float64 gradcheck). Затем — семантика QKV, backward без dX, checkpoint/streaming,
отрицательные контроли и валидность harness. CPU NF4 checkpoint — отдельно **4 случая**:
сжатая/несжатая quantization state × наличие/отсутствие dX.

JUnit разбирается программно. Отсутствующие/пропущенные случаи — UNVERIFIED, не PASS.
CPU BF16 в регрессиях не подтверждает native CUDA BF16 на T4. CPU timing-тесты —
только DEBUG-ONLY. Все вызовы pytest отключают глобальный coverage-порог subset-запуска.
Дополнительно обязательны `test_d2_mixed_precision_backward.py` (**50 CPU cases**) и
`test_d2_reference_precision.py` (**178 CPU cases**), с отдельной проверкой точного JUnit
count без skip. Они входят и в общий regression набор. Проверяются cast/scaling seams,
rounded float64 oracle и scoped reference arithmetic. Это CPU-регрессии, не CUDA-результат.


In [ ]:
PROJECTIONS = {
    "single": ["o_proj"],
    "qkv": ["q_proj", "k_proj", "v_proj"],
    "mlp": ["gate_proj", "up_proj", "down_proj"],
}
EXPECTED_FUNCTIONS = {
    "single": "_FastLoraSingleProjectionBackward",
    "qkv": "_FastLoraQKVBackward",
    "mlp": "_FastLoraSwiGLUBackward",
}


def run_pytest(
    label: str,
    files: list[str],
    *,
    device: str,
    expression: str,
    keyword: str | None = None,
    expected_count: int | None = None,
    required_classes: list[str] | None = None,
) -> dict:
    directory = artifact_dir(label)
    junit = directory / "junit.xml"
    argv = [
        PYTHON,
        "-m",
        "pytest",
        "-p",
        "pytest_cov",
        "--no-cov",
        "-o",
        "addopts=",
        "-o",
        "junit_family=legacy",
        "-v",
        "-ra",
        "--tb=long",
        "--strict-markers",
        "-m",
        expression,
        "-p",
        "no:cacheprovider",
        "--junitxml",
        str(junit),
        *files,
    ]
    if keyword:
        argv += ["-k", keyword]
    record = run_logged(label, argv, device=device, cwd=CHECKOUT, directory=directory, strict=False)
    cases = []
    parse_error = None
    if junit.is_file():
        try:
            cases = list(ET.parse(junit).getroot().iter("testcase"))
        except (ET.ParseError, OSError) as exc:
            parse_error = f"{type(exc).__name__}: {exc}"
    else:
        parse_error = "JUnit missing"
    skipped = [case.attrib for case in cases if case.find("skipped") is not None]
    failed = [
        case.attrib
        for case in cases
        if any(case.find(tag) is not None for tag in ("failure", "error"))
    ]
    summary = {
        "count": len(cases),
        "skipped": skipped,
        "failed": failed,
        "cases": [case.attrib for case in cases],
        "junit": str(junit),
        "returncode": record["returncode"],
        "command_status": record["status"],
        "command_error": record.get("error"),
        "junit_error": parse_error,
    }
    write_json(directory / "pytest-summary.json", summary)
    # Preserve available JUnit before propagating a command failure; never waive its exit.
    if record["returncode"] != 0:
        raise RuntimeError(f"Pytest exit={record['returncode']}; raw logs/summary: {directory}")
    if record["status"] != "EXIT_ZERO":
        raise RuntimeError(
            f"Pytest command/source snapshot invalid: {record['status']}; {directory}"
        )
    if parse_error is not None:
        raise UnverifiedGateError(f"JUnit unavailable/unreadable: {parse_error}; {directory}")
    if failed:
        raise RuntimeError(f"JUnit содержит ошибки: {directory}")
    if not cases or skipped:
        raise UnverifiedGateError(f"Пустой/пропущенный набор pytest: {summary}")
    if expected_count is not None and len(cases) != expected_count:
        raise UnverifiedGateError(f"Ожидалось {expected_count} случаев, собрано {len(cases)}")
    for required in required_classes or []:
        if not any(required in case.get("classname", "") for case in summary["cases"]):
            raise UnverifiedGateError(f"Gradcheck-набор не выполнен: {required}")
    return summary


def validate_parity(report: dict, request_dx: bool) -> None:
    rows = report["rows"]
    expected_count = 22 if request_dx else 19
    if len(rows) != expected_count or report.get("request_dX") != request_dx:
        raise RuntimeError("Parity rows/dX count does not match the declared gate")
    for path, projections in PROJECTIONS.items():
        group = [row for row in rows if row["path"] == path]
        expected_backward = {
            f"{gradient}/{projection}" for gradient in ("dA", "dB") for projection in projections
        }
        if request_dx:
            expected_backward.add("dX")
        if {row["quantity"] for row in group if row["phase"] == "backward"} != expected_backward:
            raise RuntimeError(f"Missing/unexpected gradients: {path}")
        expected_forward = (
            {f"Y/{projection}" for projection in projections} if path == "qkv" else {"Y/Y"}
        )
        forward = [row["quantity"] for row in group if row["phase"] == "forward"]
        if set(forward) != expected_forward or len(forward) != len(expected_forward):
            raise RuntimeError(f"Missing/duplicate/unexpected forward outputs: {path}")
        if report["patch_counts"][path] <= 0:
            raise RuntimeError(f"Fast path not installed: {path}")
        nodes = report.get("execution", {}).get(path, [])
        if len(nodes) != len(expected_forward) or any(
            node != EXPECTED_FUNCTIONS[path] for node in nodes
        ):
            raise RuntimeError(f"Fast path custom execution missing/delegated: {path}")
        if report["negative_control"][path]["changed_adapter_detected"] is not True:
            raise RuntimeError(f"Negative control failed: {path}")
    if not all(row["passed"] and row["finite"] for row in rows):
        raise RuntimeError("Parity gate has a failing or nonfinite row")


def validate_loss_reference_arithmetic(report: dict, dtype: str) -> None:
    metadata = report.get("reference_arithmetic")
    required_true = dtype in {"fp16", "bf16"}
    if (
        not isinstance(metadata, dict)
        or metadata.get("ctx_attribute") != "grad_fn.reference_order"
        or metadata.get("required_true") is not required_true
    ):
        raise RuntimeError("Reference arithmetic metadata absent/incorrect for loss precision")
    expected = metadata.get("expected_modules")
    layers = report.get("model", {}).get("num_hidden_layers")
    if (
        not isinstance(expected, dict)
        or set(expected) != {"qkv", "mlp"}
        or type(layers) is not int
        or layers <= 0
    ):
        raise RuntimeError("Reference arithmetic expected-module contract missing")
    execution = report.get("execution", {})
    for path, multiplier in (("qkv", 3), ("mlp", 1)):
        names = expected[path]
        nodes = execution.get(path)
        if (
            not isinstance(names, list)
            or not all(isinstance(name, str) for name in names)
            or len(names) != multiplier * layers
            or len(set(names)) != len(names)
            or not isinstance(nodes, dict)
            or set(nodes) != set(names)
            or any(node != EXPECTED_FUNCTIONS[path] for node in nodes.values())
        ):
            raise RuntimeError("Reference arithmetic module/execution keys incomplete")
    for row in report["rows"]:
        modes = row.get("reference_arithmetic_modes")
        if (
            row.get("reference_arithmetic_status") != "observed"
            or not isinstance(modes, dict)
            or set(modes) != set(expected)
        ):
            raise RuntimeError("Reference arithmetic per-step observation absent/incomplete")
        for path, names in expected.items():
            values = modes[path]
            if (
                not isinstance(values, dict)
                or set(values) != set(names)
                or any(
                    type(value) is not bool or (required_true and value is not True)
                    for value in values.values()
                )
            ):
                raise RuntimeError("Reference arithmetic per-step scope/order not observed")


def probe(
    label: str,
    mode: str,
    device: str,
    dtype: str,
    *,
    no_dx: bool = False,
    gradcheck: bool = False,
    shapes: str = "tiny",
) -> dict:
    directory = artifact_dir(label)
    prefix = directory / "evidence"
    argv = [
        PYTHON,
        str(HARNESS),
        mode,
        "--device",
        device,
        "--dtype",
        dtype,
        "--seed",
        str(SEED),
        "--output-prefix",
        str(prefix),
        "--shapes",
        shapes,
    ]
    if mode == "parity":
        if not gradcheck:
            argv += ["--skip-gradcheck"]
        if no_dx:
            argv += ["--no-dx"]
    elif mode == "loss":
        argv += ["--steps", str(STEPS)]
    elif mode == "timing":
        argv += ["--warmup", str(WARMUP), "--repeats", str(REPEATS), "--tokens", str(TOKENS)]
    else:
        raise ValueError(mode)
    run_logged(label, argv, device=device, cwd=CHECKOUT, directory=directory)
    json_path, csv_path = Path(str(prefix) + ".json"), Path(str(prefix) + ".csv")
    if not json_path.is_file() or not csv_path.is_file():
        raise UnverifiedGateError(f"Harness did not write JSON and CSV: {directory}")
    report = json.loads(json_path.read_text(encoding="utf-8"))
    with csv_path.open(newline="", encoding="utf-8") as handle:
        csv_rows = list(csv.DictReader(handle))
    if report.get("passed") is not True or not report.get("rows"):
        raise RuntimeError(f"Harness did not pass: {directory}")
    if len(csv_rows) != len(report["rows"]):
        raise RuntimeError("JSON/CSV row count mismatch")
    for key, value in {
        "mode": mode,
        "device": device,
        "dtype": dtype,
        "seed": SEED,
        "fixture": "SYNTHETIC",
    }.items():
        if report.get(key) != value:
            raise RuntimeError(f"Unexpected harness metadata: {key}")
    environment = report["environment"]
    if SOURCE_MODE == "zip":
        if (
            not {"git_head", "git_status"}.issubset(environment)
            or environment["git_head"] is not None
            or environment["git_status"] is not None
        ):
            raise RuntimeError("ZIP snapshot must retain honest no-Git provenance, not a Git HEAD")
        errors = environment.get("collection_errors")
        if not isinstance(errors, dict) or any(
            not isinstance(errors.get(query), str) or not errors[query].strip()
            for query in ("git rev-parse HEAD", "git status --short")
        ):
            raise RuntimeError("ZIP snapshot must record both Git query error diagnostics")
    elif environment["git_head"] != REMOTE_HEAD or environment["git_status"]:
        raise RuntimeError("Harness ran a different commit or dirty checkout")
    expected_rule_hash = MANIFEST["files_sha256"][str(RULE.relative_to(CHECKOUT))]
    if environment["decision_rule_sha256"] != expected_rule_hash:
        raise RuntimeError("Decision-rule hash changed during the run")
    expected_kernels = {"fast_lora.py", "fast_lora_qkv.py", "fast_lora_mlp.py"}
    kernel_hashes = environment.get("kernel_sha256")
    if not isinstance(kernel_hashes, dict) or set(kernel_hashes) != expected_kernels:
        raise RuntimeError("Kernel fingerprint set incomplete/unexpected")
    for filename, digest in kernel_hashes.items():
        if digest != MANIFEST["files_sha256"]["src/soup_cli/utils/" + filename]:
            raise RuntimeError(f"Kernel hash changed during the run: {filename}")
    expected_harness_hash = MANIFEST["files_sha256"].get(str(HARNESS.relative_to(CHECKOUT)))
    if not expected_harness_hash or environment.get("harness_sha256") != expected_harness_hash:
        raise RuntimeError("Harness hash missing/changed during the run")
    if mode == "parity":
        validate_parity(report, not no_dx)
        if gradcheck:
            for path, projections in PROJECTIONS.items():
                check = report["gradcheck"][path]
                variables = {"X"} | {
                    f"{matrix}/{projection}" for matrix in ("A", "B") for projection in projections
                }
                if not check["passed"] or set(check["variables"]) != variables:
                    raise RuntimeError(f"Incomplete float64 gradcheck: {path}")
    if mode == "loss":
        validate_loss_reference_arithmetic(report, dtype)
        rows = report["rows"]
        if report.get("steps") != 50 or len(rows) != 50:
            raise RuntimeError("Loss acceptance requires exactly 50 updates")
        if [row["step"] for row in rows] != list(range(1, 51)):
            raise RuntimeError("Loss step sequence incomplete")
        for row in rows:
            if not all(math.isfinite(row[key]) for key in ("baseline_loss", "fast_loss")):
                raise RuntimeError("Nonfinite loss")
            if format(row["baseline_loss"], ".3f") != format(row["fast_loss"], ".3f"):
                raise RuntimeError("Preregistered three-decimal loss gate failed")
        if not report["base_unchanged"] or not report["negative_control"]["unpatched_rejected"]:
            raise RuntimeError("Frozen base or execution negative control failed")
        if any(
            report["patch_counts"].get(path, 0) <= 0 or not report["execution"].get(path)
            for path in PROJECTIONS
        ):
            raise RuntimeError("Not all three fast paths were exercised")
    return report

In [ ]:
gate(
    "CPU_GRADCHECK",
    lambda: run_pytest(
        "cpu-existing-gradchecks",
        CORE_TESTS,
        device="cpu",
        expression="not gpu and not smoke",
        keyword="gradcheck",
        required_classes=[Path(path).stem for path in CORE_TESTS],
    ),
)
gate(
    "CPU_PARITY",
    lambda: probe("cpu-fp32-parity-gradcheck", "parity", "cpu", "fp32", gradcheck=True),
)
gate(
    "CPU_REGRESSIONS",
    lambda: run_pytest(
        "cpu-regressions",
        TEST_FILES,
        device="cpu",
        expression="not gpu and not smoke",
        keyword="not mps",
    ),
)
gate(
    "CPU_NF4_CHECKPOINT",
    lambda: run_pytest(
        "cpu-nf4-checkpoint",
        ["tests/test_d2_checkpoint_and_dtypes.py"],
        device="cpu",
        expression="not gpu and not smoke",
        keyword="nf4_single_non_reentrant_checkpoint",
        expected_count=4,
    ),
)
gate(
    "CPU_MIXED_PRECISION",
    lambda: run_pytest(
        "cpu-mixed-precision",
        ["tests/test_d2_mixed_precision_backward.py"],
        device="cpu",
        expression="not gpu and not smoke",
        keyword="not mps",
        expected_count=50,
    ),
)
gate(
    "CPU_REFERENCE_PRECISION",
    lambda: run_pytest(
        "cpu-reference-precision",
        ["tests/test_d2_reference_precision.py"],
        device="cpu",
        expression="not gpu and not smoke",
        keyword="not mps",
        expected_count=178,
    ),
)
say("CPU gates PASS. CUDA пока не подтверждена. Для продолжения выберите T4 в Kaggle UI.")

## 6. Вручную включённая T4: preflight и hardware blocks

Эта секция не включает GPU через API. При `RUN_CUDA=False` CUDA остаётся UNVERIFIED.
Проверяется реальная T4, compute capability 7.5, один **видимый** CUDA device, CUDA build
и небольшая allocation. Сохраняются card/UUID/VRAM/build/driver/clocks/power/peer-processes.
Никакие чужие процессы не завершаются.

**Native BF16 и current Triton/Liger/Unsloth на T4 — UNVERIFIED_HARDWARE_BLOCK**, не
SKIP/PASS. Старые GPU NF4-тесты QKV/MLP используют BF16; старый `TestMicroBenchmark`
использует большую BF16-форму до correctness gate. Поэтому здесь нет общего
`pytest tests/ -m gpu`: на T4 он был бы неверным способом проверить заданный протокол.
Отдельно выполняются новые **4 fp16 CUDA NF4 single-checkpoint** случая.

In [ ]:
GPU_PROBE = r"""
import importlib.metadata
import json
import os
import sys
from pathlib import Path
from rich.console import Console

def main():
    assert os.environ["CUDA_VISIBLE_DEVICES"] == "0"
    import torch
    assert torch.cuda.is_available(), "CUDA unavailable: select free T4 in the Kaggle UI"
    assert torch.cuda.device_count() == 1, "Exactly one CUDA device must be visible"
    capability = list(torch.cuda.get_device_capability(0))
    props = torch.cuda.get_device_properties(0)
    name = torch.cuda.get_device_name(0)
    assert "T4" in name and capability == [7, 5], f"Not the declared T4 target: {name} {capability}"
    assert torch.version.cuda is not None, "Installed Torch is a CPU-only build"
    assert (torch.ones(4, device="cuda") + 1).sum().item() == 8
    torch.cuda.synchronize()
    report = {
        "card": name, "compute_capability": capability, "uuid": str(getattr(props, "uuid", "")),
        "total_memory_bytes": props.total_memory, "visible_device_count": 1,
        "cuda_visible_devices": os.environ["CUDA_VISIBLE_DEVICES"],
        "python": sys.version, "torch": torch.__version__, "cuda_build": torch.version.cuda,
        "versions": {name: importlib.metadata.version(name) for name in (
            "peft", "transformers", "bitsandbytes", "accelerate",
        )}, "native_bf16": "UNVERIFIED_HARDWARE_BLOCK",
        "triton_gate": "UNVERIFIED_HARDWARE_BLOCK",
    }
    Path(sys.argv[1]).write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    Console().print(report)

main()
"""


def cuda_preflight() -> dict:
    require_gates(*GATE_DEPENDENCIES["CUDA_PREFLIGHT"])
    assert_checkout()
    smi_snapshot("cuda-preflight-before")
    path = RUN / "cuda-hardware.json"
    try:
        run_logged("cuda-preflight", [PYTHON, "-c", GPU_PROBE, str(path)], device="cuda")
    except RuntimeError as exc:
        raise UnverifiedGateError(
            "CUDA preflight не подтверждён; сохранены логи hardware/import error"
        ) from exc
    finally:
        smi_snapshot("cuda-preflight-after")
    return json.loads(path.read_text(encoding="utf-8"))


if RUN_CUDA:
    CUDA_HARDWARE = gate("CUDA_PREFLIGHT", cuda_preflight)
else:
    say("RUN_CUDA=False: CUDA не запускалась, результаты остаются UNVERIFIED.")
    save_gates()

## 7. CUDA correctness: fp32/fp16 dense parity → loss 50 → NF4 checkpoint

Dense parity выполняется для обеих точностей **с dX и без dX**. fp32 использует критерий
правила; fp16 сравнивается с float64 oracle. Проверяется каждая dA/dB, отсутствие
градиентов frozen bases, фактический custom autograd и отрицательный контроль.

Loss: одинаковые начальные веса, фиксированные SYNTHETIC токены, seed **792**, **50**
обновлений. Сохраняется вся кривая и неокруглённая разница; gate — одинаковый формат
`.3f` на каждом шаге, не bit-exactness. Обе точности проверяются отдельно. Ошибка, в том
числе на округлении, сохраняется как FAIL; менять правило постфактум нельзя.

NF4 **не реализована в harness**. Здесь только отдельный pytest на реальных packed
weights/state и `checkpoint(use_reentrant=False)`. Его PASS не означает, что CUDA NF4
QKV/MLP/streamed пути или native BF16 подтверждены.
Loss JSON обязан содержать `reference_arithmetic` metadata и **на каждом шаге**
`reference_arithmetic_modes`/`reference_arithmetic_status`. Для fp16 (и требуемого bf16
на другом разрешённом hardware) каждое объявленное QKV/MLP наблюдение обязано быть
**literal True** для `grad_fn.reference_order`; одно имя custom Function этого не доказывает.
fp32 записывает literal bool, но не требует True. Отсутствие/частичный/устаревший mode
блокирует PASS. Не меняются seed 792, 50 шагов или `.3f`; все failed rows остаются в logs/JSON/CSV.


In [ ]:
def cuda_dense_parity() -> None:
    require_gates("CUDA_PREFLIGHT")
    assert_checkout()
    for dtype in ("fp32", "fp16"):
        for no_dx in (False, True):
            label = f"cuda-{dtype}-parity-{'no-dx' if no_dx else 'dx'}"
            smi_snapshot(label + "-before")
            try:
                probe(label, "parity", "cuda", dtype, no_dx=no_dx)
            finally:
                smi_snapshot(label + "-after")


def cuda_loss() -> None:
    require_gates("CUDA_DENSE_PARITY")
    for dtype in ("fp32", "fp16"):
        label = f"cuda-{dtype}-loss-50"
        smi_snapshot(label + "-before")
        try:
            probe(label, "loss", "cuda", dtype)
        finally:
            smi_snapshot(label + "-after")


def cuda_nf4_checkpoint() -> dict:
    require_gates("CUDA_LOSS_50")
    smi_snapshot("cuda-nf4-checkpoint-before")
    try:
        return run_pytest(
            "cuda-fp16-nf4-checkpoint",
            ["tests/test_d2_checkpoint_and_dtypes.py"],
            device="cuda",
            expression="gpu and not smoke",
            keyword="nf4_single_non_reentrant_checkpoint",
            expected_count=4,
        )
    finally:
        smi_snapshot("cuda-nf4-checkpoint-after")


if RUN_CUDA:
    gate("CUDA_DENSE_PARITY", cuda_dense_parity)
    gate("CUDA_LOSS_50", cuda_loss)
    gate("CUDA_NF4_SINGLE_CHECKPOINT_FP16", cuda_nf4_checkpoint)
else:
    say("CUDA correctness не запускалась: UNVERIFIED, не PASS.")

## 8. Только после зелёного correctness: ABBA, CUDA events, raw samples

Маленький режим: fp32 и fp16, single/QKV/MLP, dX и no-dX, warmup **5**, **5 ABBA раундов**,
**16 токенов**. Один раунд = A(PEFT), B(fast), B(fast), A(PEFT). Отдельные CUDA events
для forward/backward, peak allocated и reserved VRAM — разные поля.

В текущем CLI ABBA-руки работают **в одном процессе**, а не в свежем процессе на руку.
Каждый вызов harness — отдельный процесс. Память включает одновременно оба arm/model;
это не isolated-arm VRAM и не end-to-end benchmark. Правило предпочитает fresh-process
руки, поэтому ограничение явно сохраняется. `nvidia-smi` и peer-проверки выполняются
вне timed regions; harness также пишет per-arm validity. Любая загрязнённая рука —
**VOID**, raw данные остаются. Нет выборки clocks/power/peers — **NO VERDICT**.
Даже exit=0 не даёт автоматического performance PASS или D2 Done.

`RUN_8B_SHAPED_BLOCK=True` отдельно разрешает только SYNTHETIC single-block размеры
`llama3.1-8b` в fp16; не загружает и не тренирует полную 8B-модель. Возможный OOM сохраняется.

In [ ]:
def collect_timing(name: str, shapes: str, dtypes: tuple[str, ...]) -> None:
    begin_gate(name)
    summaries = []
    try:
        require_gates(*GATE_DEPENDENCIES[name])
        assert_checkout()
        for dtype in dtypes:
            label = f"cuda-{dtype}-timing-{'tiny' if shapes == 'tiny' else '8b-block'}"
            before = smi_snapshot(label + "-before")
            try:
                report = probe(label, "timing", "cuda", dtype, shapes=shapes)
            finally:
                after = smi_snapshot(label + "-after")
            rows = report["rows"]
            if report["timer"] != "CUDA events" or report["round_order"] != "ABBA":
                raise RuntimeError("Requested CUDA-event ABBA timing was not collected")
            if report["shapes"] != shapes or len(rows) != 3 * 2 * REPEATS * 4:
                raise RuntimeError("Timing shape/sample count mismatch")
            if not report["correctness_gate"]["passed"]:
                raise RuntimeError("Timing shape-specific parity gate did not pass")
            for path in PROJECTIONS:
                for dx in (False, True):
                    for round_index in range(1, REPEATS + 1):
                        arms = [
                            row
                            for row in rows
                            if row["path"] == path
                            and row["request_dX"] == dx
                            and row["round"] == round_index
                        ]
                        if [row["arm"] for row in arms] != ["A", "B", "B", "A"]:
                            raise RuntimeError("ABBA sample order incomplete")
            validity = [row["validity"] for row in rows]
            summaries.append(
                {
                    "dtype": dtype,
                    "shapes": shapes,
                    "rows": len(rows),
                    "void_rows": sum(sample.get("status") == "VOID" for sample in validity),
                    "missing_validity_rows": sum(
                        not sample.get("available") or not sample.get("target_uuid")
                        for sample in validity
                    ),
                    "smi_snapshot_commands_ok": all(
                        record["returncode"] == 0
                        for samples in (before, after)
                        for record in samples.values()
                    ),
                    "timing_verdict": report["timing_verdict"],
                    "process_isolation": report.get("process_isolation"),
                    "memory_scope": report.get("memory_scope"),
                }
            )
    except BaseException as exc:
        set_gate_status(
            name,
            {
                "status": "UNVERIFIED" if isinstance(exc, UnverifiedGateError) else "FAIL",
                "error": f"{type(exc).__name__}: {exc}",
                "completed_samples": summaries,
            },
        )
        save_gates()
        raise
    set_gate_status(
        name,
        {
            "status": "COLLECTED_NO_VERDICT",
            "samples": summaries,
            "reason": "Review raw per-arm clocks/power/peers; no speedup claim",
        },
    )
    save_gates()


if RUN_CUDA and RUN_TINY_TIMING:
    collect_timing("CUDA_TINY_TIMING", "tiny", ("fp32", "fp16"))
if RUN_CUDA and RUN_8B_SHAPED_BLOCK:
    collect_timing("CUDA_8B_SHAPED_BLOCK", "llama3.1-8b", ("fp16",))
if not RUN_CUDA:
    say("GPU timing не запускался. CPU_PASS / CUDA_UNVERIFIED — только частичный результат.")

## 9. Итог и локальный ZIP для скачивания (можно после ошибки)

Выполните эту ячейку отдельно после сбоя, не повторяя неудачный эксперимент. Она ничего
не отправляет на сервер и не делает публичным. ZIP содержит только текущую папку
`d2-evidence/RUN_ID`, **не** venv, checkout, Kaggle input datasets или переменные окружения.
Предыдущие успешные/неуспешные RUN_ID остаются на диске. Скачайте ZIP через Files/Output
или ссылку ниже; проверьте содержимое вручную перед добровольной передачей ревьюеру.

При shutdown/сбое самой Kaggle-сессии несохранённые файлы могут исчезнуть — скачивайте
архив до окончания сессии. Ревьюеру нужны `gate_status.json`, source manifest,
command ledger, stdout/stderr, все JSON/CSV и JUnit, `nvidia-smi` и per-arm validity.
CPU PASS, четыре NF4 single-checkpoint PASS или собранный timing **не закрывают D2**:
native BF16, Triton, CUDA NF4 QKV/MLP/streaming остаются неподтверждёнными; T00/F1 и
решение Done принадлежат команде. Не создавайте комментарии/карточки автоматически.

Перед чтением/hash/архивом проверяется вся текущая папка. Symlink (даже внутренний или
сломанный), junction/reparse point, hardlink, `..` и выход за RUN блокируют ZIP. Архив
содержит только проверенные файлы; существующий ZIP не перезаписывается.


In [ ]:
from IPython.display import FileLink, display


def artifact_files() -> list[Path]:
    safe_artifact_path(RUN, WORK)
    files = []

    def walk_error(error):
        raise error

    for directory, subdirectories, filenames in os.walk(
        RUN,
        followlinks=False,
        onerror=walk_error,
    ):
        safe_artifact_path(Path(directory), RUN)
        for name in subdirectories + filenames:
            path = Path(directory) / name
            metadata = safe_artifact_path(path, RUN)
            if stat.S_ISREG(metadata.st_mode):
                files.append(path)
    return sorted(files)


def artifact_bytes(path: Path) -> bytes:
    before = safe_artifact_path(path, RUN)
    descriptor = os.open(
        path, os.O_RDONLY | getattr(os, "O_BINARY", 0) | getattr(os, "O_NOFOLLOW", 0)
    )
    with os.fdopen(descriptor, "rb") as handle:
        opened = os.fstat(handle.fileno())
        if (
            not stat.S_ISREG(opened.st_mode)
            or opened.st_nlink != 1
            or (opened.st_dev, opened.st_ino) != (before.st_dev, before.st_ino)
        ):
            raise RuntimeError(f"Unsafe artifact changed before reading: {path}")
        data = handle.read()
    after = safe_artifact_path(path, RUN)
    if (after.st_dev, after.st_ino, after.st_size, after.st_mtime_ns) != (
        opened.st_dev,
        opened.st_ino,
        opened.st_size,
        opened.st_mtime_ns,
    ):
        raise RuntimeError(f"Unsafe artifact changed while reading: {path}")
    return data


# Validate the ENTIRE tree before reading logs, hashing, or writing summaries.
artifact_files()

for name, result in list(GATES.items()):
    if result["status"] == "RUNNING":
        set_gate_status(
            name, {"status": "UNVERIFIED", "reason": "Stage interrupted; inspect raw logs"}
        )
save_gates()
command_ledger = RUN / "commands.jsonl"
records = (
    [
        json.loads(line)
        for line in command_ledger.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    if command_ledger.exists()
    else []
)
write_json(
    RUN / "artifact-index.json",
    {
        "run_id": RUN_ID,
        "commands": len(records),
        "nonzero_or_missing_exit_codes": [
            record for record in records if record["returncode"] != 0
        ],
        "overall": "PARTIAL / REVIEW REQUIRED — never automatic D2 acceptance",
        "gates": GATES,
    },
)
files = [path for path in artifact_files() if path != RUN / "artifact-sha256.json"]
manifest = {
    Path(os.path.relpath(path, RUN)).as_posix(): hashlib.sha256(artifact_bytes(path)).hexdigest()
    for path in files
}
write_json(RUN / "artifact-sha256.json", manifest)
if not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_ID):
    raise RuntimeError("Unsafe archive RUN_ID")
archive = str(WORK / ("d2-evidence-" + RUN_ID + ".zip"))
files = artifact_files()
# Explicit, validated members only; no make_archive traversal or link following.
# Exclusive creation refuses existing archive/link instead of overwriting it.
with zipfile.ZipFile(archive, "x", compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in files:
        relative = Path(os.path.relpath(path, RUN)).as_posix()
        data = artifact_bytes(path)
        if path != RUN / "artifact-sha256.json":
            if hashlib.sha256(data).hexdigest() != manifest[relative]:
                raise RuntimeError(f"Artifact changed after hashing: {path}")
        bundle.writestr(RUN_ID + "/" + relative, data)
say(json.dumps(GATES, indent=2, ensure_ascii=False))
say(f"ZIP для ручного скачивания: {archive}; ничего не опубликовано и не загружено.")
display(FileLink(archive))